# A2.7 · Attribution: an audit trail that answers "who"

**Function A — Securing AI Architectures → Securing the Architecture — Identity and Ingress**  ·  *Security of AI*

Builds on **[A2.6 · Ingress: marking untrusted content at the door](https://spbreed.github.io/cyber-commons/lessons/A2.6.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry, Sigstore |

## What this lesson is

**What it covers.** Answer 'which user caused this deletion' from the trace, then try the same on a trace missing one field.

**Why a security engineer needs it.** Without the motivating input, root cause cannot be established at all; without the principal, nothing can be attributed. The control it builds is: per-hop attribution written to an append-only store outside the agent's reach.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A trace that records tool calls is not evidence. Evidence answers who asked, which agent acted, what authority it held, and what input made it act — and an auditor will ask all four in that order.

> **At CyberTravels.** The four fields an auditor will ask about that $5,000 refund: which traveller, which agent, under what authority, and what text made it act. CyberTravels currently records the third and a version of the second. R11.

## 2 · The framework

```
   the four fields an auditor asks for, in order

   1. principal      dana@corp          who asked
   2. agent          patch-agent#7      who acted
   3. authority      repo:write, exp+90s   under what
   4. motivating input   PR #412 body, [data]   why

   a trace with 1-3 and not 4 cannot distinguish authorised from injected
```

**Mitigates: T8 Repudiation & Untraceability · T13 Rogue Agents.**

A1.14 showed a log that was complete for debugging and empty for investigation.
This is the record that is not.

Four fields, each answering a question the tool-call log could not:

**The human principal** — who caused this. From A2.1.

**The agent identity and instance** — what performed it, and which run. From
A2.2, so it is attested rather than claimed.

**The delegation chain** — how authority got from the human to this action. From
A2.3, which is also what makes A1.17's laundering path visible.

**The motivating input, with its origin** — what made the agent decide. From
A2.6. This is the field that establishes root cause, and the one most often
missing, because logging tool calls feels like logging decisions.

Then the structural property, which is not a field: **the store must be outside
the agent's reach.** An agent with broad credentials can usually touch the
logging stack, and a record the actor can edit is not evidence. Append-only,
different credential, ideally different trust domain.

The test is not whether the log looks thorough. It is whether you can answer
"which user caused this, and what made it happen" without asking anyone.

> **What this control closes.**
>
> Makes the incident answerable. Also the control an auditor asks for first, because a system that cannot attribute an action cannot be defended even on a quiet day.

## 3 · The check, as a skill

The four questions an auditor asks about that $5,000 refund have to be answerable from one entry. The skill puts them to a single ledger record, and then tries to amend the record as the agent — because a complete log its subject can edit records what the agent wanted you to see.

### The skill — [`skills/identity/attribution-ledger-check/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/identity/attribution-ledger-check/SKILL.md)

```yaml
name: attribution-ledger-check
description: >-
  Check that one record answers all four investigation questions — human
  principal, attested workload and run, delegation chain, and the motivating
  input with its origin — and that the agent cannot amend it. Use when
  designing agent audit records rather than reading them.
allowed-tools: Read, Grep, Glob
```

# One entry, four questions

An audit trail is not "we log tool calls". It is a record that answers, from a
single entry, the four questions an investigation asks — and that the subject of
the record cannot edit. Both halves are required: a complete record the agent
can rewrite is a record of what the agent wanted you to see.

## When to use this

While designing the record. Retrofitting attribution after an incident means
reconstructing it from four services' timestamps, which does not happen at 2am.

## Procedure

**1 — Write the four questions down first.** Which human. Which workload and
which run. Through what delegation chain. On what motivating input, from what
origin. Design the entry to answer them; do not collect fields and hope.

**2 — Map each question to a field.** Principal from the delegated token, not
from a header the agent set. Workload and instance from the attestation. Chain
from the token's `act` nesting. Motivating input from the ingress record that
caused this step, with its origin tag.

**3 — Test each question against one entry.** Alone. If answering needs a join
across services, that question is unanswered for practical purposes and should
be recorded as such.

**4 — Attempt the amendment as the agent.** Append, overwrite, delete, reorder.
Every one must be refused by something the agent does not control — an
append-only store, a separate writer identity, a downstream sink it cannot
reach.

**5 — Check the write path's own identity.** If the agent's role can write to
the log destination, the ledger is advisory whatever the API says.

## Output contract

```json
{
  "entry": {"principal": "str", "workload": "str", "instance": "str",
            "chain": ["str"], "motivating_input": {"ref": "str", "origin": "str"}},
  "questions": [{"question": "str", "answered_from_single_entry": true}],
  "amendment_attempts": [{"operation": "append|overwrite|delete|reorder", "refused_by": "str"}],
  "writer_identity": {"agent_can_write_destination": false}
}
```

## Failure modes

- **A principal field the agent populates.** It is a claim, not attribution.
- **Answering a question by correlation.** Correlation is a plan, not a record.
- **Testing amendment through the API only.** Try the storage layer the agent's
  role can reach.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/identity/attribution-ledger-check/scripts/attribution_ledger_check.py
SCRIPT = "skills/identity/attribution-ledger-check/scripts/attribution_ledger_check.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

One ledger entry answers all four investigation questions — the human principal, the attested workload and run, the delegation chain, and the motivating input with its origin — and the agent's attempt to amend the record is refused.

## Your turn

Take the last significant action one of your agents performed and try to fill in these four fields from what you actually logged. The missing one is almost always the motivating input.

---

**Next → [A2.8 · An audit trail the workload cannot forge](https://spbreed.github.io/cyber-commons/lessons/A2.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*